# Modelo Predictivo de Derivación/Traslado al Alta — GRD Chile 2024

**Objetivo de investigación:** Identificar qué factores clínicos, administrativos e institucionales
predicen que un paciente sea dado de alta mediante **derivación o traslado** a otro establecimiento,
en lugar de ser egresado a domicilio o por alta voluntaria.

**¿Por qué excluir los FALLECIDOS?**  
Los pacientes fallecidos representan una población con un mecanismo de egreso completamente distinto.
Incluirlos mezclaría dos preguntas clínicas diferentes: "¿quién muere?" vs. "¿quién requiere
ser derivado a otro centro?". Esta separación mantiene la coherencia conceptual del modelo
y evita que variables de gravedad extrema dominen incorrectamente los coeficientes.

**¿Por qué COD_HOSPITAL es la variable central?**  
La decisión de derivar no depende solo del paciente — depende fuertemente del establecimiento:
su nivel de complejidad, su capacidad resolutiva, su red de derivación y sus patrones institucionales.
Un hospital de baja complejidad derivará más que un hospital de alta complejidad, incluso para
pacientes con perfil clínico similar. Identificar y cuantificar este efecto institucional
es el núcleo de la investigación.

**Estructura:**
1. Setup
2. Carga de datos
3. Variable objetivo
4. Feature Engineering
5. Análisis Exploratorio (EDA)
6. Preparación del dataset
7. Modelado (LR L2 + LR L1)
8. Evaluación
9. Odds Ratios con IC 95%
10. Análisis por hospital: tasa real vs. predicha
11. Exportación
12. Resumen y limitaciones

## 0. Setup e imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, f1_score
)

try:
    from category_encoders import TargetEncoder
    TARGET_ENC_AVAILABLE = True
    print('category_encoders disponible — TargetEncoder para COD_HOSPITAL')
except ImportError:
    TARGET_ENC_AVAILABLE = False
    print('category_encoders no disponible — se usará frequency encoding')

PLOTS_DIR   = Path('../plots')
MODELS_DIR  = Path('../models')
DATA_DIR    = Path('../data/processed')
for d in [PLOTS_DIR, MODELS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
print('Setup completo.')

## 1. Carga de datos

In [2]:
DATA_PATH = '../data/archivosDuros/GRD_PUBLICO_2024.txt'

print(f'Leyendo {DATA_PATH} ...')
df_raw = pd.read_csv(
    DATA_PATH,
    sep='|',
    encoding='latin1',
    dtype=str,
    low_memory=False
)
print(f'Shape original: {df_raw.shape}')

# Eliminar fila duplicada de headers
if (df_raw.iloc[0] == df_raw.columns).mean() > 0.5:
    print('Fila duplicada de headers detectada — eliminando...')
    df_raw = df_raw.iloc[1:].reset_index(drop=True)

df = df_raw.copy()
print(f'Shape final: {df.shape}')
print('\nColumnas:', list(df.columns))

Leyendo ../data/archivosDuros/GRD_PUBLICO_2024.txt ...
Shape original: (1085813, 129)
Shape final: (1085813, 129)

Columnas: ['COD_HOSPITAL', 'ID_BENEFICIARIO', 'SEXO', 'FECHA_NACIMIENTO', 'ETNIA', 'PROVINCIA', 'COMUNA', 'NACIONALIDAD', 'PREVISION', 'SERVICIO_SALUD', 'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'ESPECIALIDAD_MEDICA', 'TIPO_ACTIVIDAD', 'FECHA_INGRESO', 'SERVICIOINGRESO', 'FECHATRASLADO1', 'SERVICIOTRASLADO1', 'FECHATRASLADO2', 'SERVICIOTRASLADO2', 'FECHATRASLADO3', 'SERVICIOTRASLADO3', 'FECHATRASLADO4', 'SERVICIOTRASLADO4', 'FECHATRASLADO5', 'SERVICIOTRASLADO5', 'FECHATRASLADO6', 'SERVICIOTRASLADO6', 'FECHATRASLADO7', 'SERVICIOTRASLADO7', 'FECHATRASLADO8', 'SERVICIOTRASLADO8', 'FECHATRASLADO9', 'SERVICIOTRASLADO9', 'FECHAALTA', 'SERVICIOALTA', 'TIPOALTA', 'CONDICIONDEALTANEONATO1', 'PESORN1', 'SEXORN1', 'RN1ESTADO', 'CONDICIONDEALTANEONATO2', 'PESORN2', 'SEXORN2', 'RN2ESTADO', 'CONDICIONDEALTANEONATO3', 'PESORN3', 'SEXORN3', 'RN3ESTADO', 'CONDICIONDEALTANEONATO4', 'PESORN4', 'SE

In [3]:
print('Distribución TIPOALTA:')
print(df['TIPOALTA'].value_counts(dropna=False).to_string())

Distribución TIPOALTA:
TIPOALTA
DOMICILIO                                        967852
HOSPITALIZACIÓN DOMICILIARIA                      36732
FALLECIDO                                         26682
DERIVACIÓN OTRO HOSPITAL DEL SERVICIO             24923
ALTA VOLUNTARIA                                   10651
DERIVACIÓN OTRO HOSPITAL DE LA RED NACIONAL        7938
DERIVACIÓN A OTROS CENTROS (CÁRCEL, HOGAR DE       3745
FUGA DEL PACIENTE                                  3299
DERIVACIÓN INST. PRIVADA (COMPRA DE SERVICIOS      2865
DERIVACIÓN INST. PRIVADA (VOLUNTARIO)              1126


## 2. Variable objetivo: TRASLADADO

Se identifica cualquier egreso cuya modalidad de alta contenga la palabra **'DERIVACIÓN'**.
Esto abarca:
- Derivación a otro hospital del servicio
- Derivación a otro hospital de la red nacional
- Derivación a otros centros (cárcel, hogar, etc.)
- Derivación a institución privada (compra de servicios o voluntario)

Los registros `FALLECIDO` se **excluyen completamente** del análisis, ya que corresponden
a una población con una dinámica clínica y administrativa distinta.

In [4]:
df['_TIPOALTA_NORM'] = df['TIPOALTA'].str.strip().str.upper()

# Excluir fallecidos
n_antes = len(df)
df = df[df['_TIPOALTA_NORM'] != 'FALLECIDO'].copy()
n_fallecidos = n_antes - len(df)
print(f'Registros excluidos (FALLECIDO): {n_fallecidos:,}')
print(f'Registros restantes: {len(df):,}')

# Variable objetivo
df['TRASLADADO'] = df['_TIPOALTA_NORM'].str.contains('DERIVACI', na=False, regex=False).astype(int)

n_total   = len(df)
n_trasl   = df['TRASLADADO'].sum()
pct       = 100 * n_trasl / n_total
print(f'\nTotal (sin fallecidos): {n_total:,}')
print(f'Trasladados (TRASLADADO=1): {n_trasl:,} ({pct:.2f}%)')
print(f'No trasladados (TRASLADADO=0): {n_total - n_trasl:,} ({100 - pct:.2f}%)')

Registros excluidos (FALLECIDO): 26,682
Registros restantes: 1,059,131

Total (sin fallecidos): 1,059,131
Trasladados (TRASLADADO=1): 40,597 (3.83%)
No trasladados (TRASLADADO=0): 1,018,534 (96.17%)


In [5]:
# Desglose de tipos de derivación
print('Tipos de derivación identificados:')
mask_deriv = df['TRASLADADO'] == 1
print(df.loc[mask_deriv, 'TIPOALTA'].value_counts().to_string())

Tipos de derivación identificados:
TIPOALTA
DERIVACIÓN OTRO HOSPITAL DEL SERVICIO            24923
DERIVACIÓN OTRO HOSPITAL DE LA RED NACIONAL       7938
DERIVACIÓN A OTROS CENTROS (CÁRCEL, HOGAR DE      3745
DERIVACIÓN INST. PRIVADA (COMPRA DE SERVICIOS     2865
DERIVACIÓN INST. PRIVADA (VOLUNTARIO)             1126


## 3. Feature Engineering

Se construyen 15 variables predictoras.

### 3.1 Fechas base y EDAD

In [6]:
def parse_date_col(series):
    return pd.to_datetime(series.str.strip(), format='%Y-%m-%d', errors='coerce')

df['_FECHA_NAC']  = parse_date_col(df['FECHA_NACIMIENTO'])
df['_FECHA_ING']  = parse_date_col(df['FECHA_INGRESO'])
df['_FECHA_ALTA'] = parse_date_col(df['FECHAALTA'])

df['EDAD'] = ((df['_FECHA_ING'] - df['_FECHA_NAC']).dt.days / 365.25).round(1)
df.loc[df['EDAD'] < 0,   'EDAD'] = np.nan
df.loc[df['EDAD'] > 120, 'EDAD'] = np.nan

mediana_edad = df['EDAD'].median()
n_imp = df['EDAD'].isna().sum()
df['EDAD'] = df['EDAD'].fillna(mediana_edad)
print(f'EDAD — nulos imputados: {n_imp:,} con mediana {mediana_edad:.1f} años')
print(df['EDAD'].describe())

EDAD — nulos imputados: 41 con mediana 46.0 años
count    1.059131e+06
mean     4.547540e+01
std      2.538699e+01
min      0.000000e+00
25%      2.620000e+01
50%      4.600000e+01
75%      6.720000e+01
max      1.089000e+02
Name: EDAD, dtype: float64


### 3.2 SEXO_BIN

In [7]:
df['_SEXO'] = df['SEXO'].str.strip().str.upper()
print('Valores SEXO:', df['_SEXO'].value_counts(dropna=False).to_dict())

n_antes = len(df)
df = df[df['_SEXO'].isin(['HOMBRE', 'MUJER'])].copy()
print(f'Registros eliminados (SEXO DESCONOCIDO): {n_antes - len(df):,}')

df['SEXO_BIN'] = (df['_SEXO'] == 'MUJER').astype(int)

Valores SEXO: {'MUJER': 620277, 'HOMBRE': 438740, 'DESCONOCIDO': 114}
Registros eliminados (SEXO DESCONOCIDO): 114


### 3.3 DIAS_HOSPITALIZACION

In [8]:
df['DIAS_HOSPITALIZACION'] = (df['_FECHA_ALTA'] - df['_FECHA_ING']).dt.days
df['DIAS_HOSPITALIZACION'] = df['DIAS_HOSPITALIZACION'].clip(lower=0).fillna(0).astype(int)
print('DIAS_HOSPITALIZACION:')
print(df['DIAS_HOSPITALIZACION'].describe())

DIAS_HOSPITALIZACION:
count    1.059017e+06
mean     5.485182e+00
std      1.154605e+01
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      6.000000e+00
max      6.610000e+02
Name: DIAS_HOSPITALIZACION, dtype: float64


### 3.4 Conteos de traslados internos, diagnósticos y procedimientos

`N_TRASLADOS_INTERNOS` es especialmente relevante para esta variable objetivo:
un paciente que ya fue trasladado internamente varias veces dentro del establecimiento
tiene mayor probabilidad de requerir derivación a otro centro al alta.

In [9]:
def count_notnull_prefix(df, prefix, n_max):
    cols = [c for c in [f'{prefix}{i}' for i in range(1, n_max + 1)] if c in df.columns]
    if not cols:
        return pd.Series(0, index=df.index)
    return df[cols].apply(lambda c: c.str.strip().replace('', np.nan)).notna().sum(axis=1)

df['N_TRASLADOS_INTERNOS'] = count_notnull_prefix(df, 'FECHATRASLADO', 9)
df['N_DIAGNOSTICOS']       = count_notnull_prefix(df, 'DIAGNOSTICO', 35)
df['N_PROCEDIMIENTOS']     = count_notnull_prefix(df, 'PROCEDIMIENTO', 30)

for col in ['N_TRASLADOS_INTERNOS', 'N_DIAGNOSTICOS', 'N_PROCEDIMIENTOS']:
    print(f'{col}: min={df[col].min()}, max={df[col].max()}, media={df[col].mean():.2f}')

N_TRASLADOS_INTERNOS: min=0, max=9, media=0.23
N_DIAGNOSTICOS: min=1, max=35, media=5.60
N_PROCEDIMIENTOS: min=0, max=30, media=7.77


### 3.5 GRD_SEVERIDAD y GRD_PESO

In [10]:
sev_col  = next((c for c in df.columns if 'SEVERIDAD' in c.upper()), None)
peso_col = next((c for c in df.columns if 'PESO' in c.upper() and '29301' in c.upper()), None)

print(f'Columna severidad : {sev_col}')
print(f'Columna peso GRD  : {peso_col}')

def to_ordinal_grd(series):
    return pd.to_numeric(series.str.strip(), errors='coerce').fillna(0).astype(int).clip(0, 3)

df['GRD_SEVERIDAD'] = to_ordinal_grd(df[sev_col]) if sev_col else 0
print('GRD_SEVERIDAD:', df['GRD_SEVERIDAD'].value_counts().sort_index().to_dict())

if peso_col:
    df['GRD_PESO'] = pd.to_numeric(
        df[peso_col].str.strip().str.replace(',', '.', regex=False),
        errors='coerce'
    ).fillna(0.0)
else:
    df['GRD_PESO'] = 0.0

print('GRD_PESO — media:', df['GRD_PESO'].mean().round(3))

Columna severidad : IR_29301_SEVERIDAD
Columna peso GRD  : IR_29301_PESO
GRD_SEVERIDAD: {0: 212038, 1: 386596, 2: 266360, 3: 194023}
GRD_PESO — media: 0.926


### 3.6 TIPO_INGRESO — one-hot

In [11]:
tipo_ing_col = next((c for c in df.columns if 'TIPO_INGRESO' in c.upper() or 'TIPOINGRESO' in c.upper()), None)
print(f'Columna tipo ingreso: {tipo_ing_col}')

if tipo_ing_col:
    df['_TIPO_INGRESO'] = df[tipo_ing_col].str.strip().str.upper()
    tipo_dummies = pd.get_dummies(df['_TIPO_INGRESO'], prefix='TIPO_INGRESO', dtype=int)
    df = pd.concat([df, tipo_dummies], axis=1)
    TIPO_ING_COLS = list(tipo_dummies.columns)
    print('Valores:', df['_TIPO_INGRESO'].value_counts().to_dict())
else:
    TIPO_ING_COLS = []
print('Columnas creadas:', TIPO_ING_COLS)

Columna tipo ingreso: TIPO_INGRESO
Valores: {'URGENCIA': 526291, 'PROGRAMADA': 387771, 'OBSTETRICA': 144912, 'DESCONOCIDO': 43}
Columnas creadas: ['TIPO_INGRESO_DESCONOCIDO', 'TIPO_INGRESO_OBSTETRICA', 'TIPO_INGRESO_PROGRAMADA', 'TIPO_INGRESO_URGENCIA']


### 3.7 PREVISION_SIMPLIF — one-hot

In [12]:
prev_col = next((c for c in df.columns if 'PREVISION' in c.upper()), None)
print(f'Columna previsión: {prev_col}')

def simplificar_prevision(val):
    if pd.isna(val):
        return 'OTRO'
    v = str(val).strip().upper()
    if 'FONASA' in v and 'LIBRE' in v:
        return 'FONASA_LIBRE'
    if 'FONASA' in v:
        return 'FONASA_MAI'
    if 'ISAPRE' in v:
        return 'ISAPRE'
    return 'OTRO'

if prev_col:
    df['PREVISION_SIMPLIF'] = df[prev_col].apply(simplificar_prevision)
    print('Distribución:', df['PREVISION_SIMPLIF'].value_counts().to_dict())
    prev_dummies = pd.get_dummies(df['PREVISION_SIMPLIF'], prefix='PREV', dtype=int)
    df = pd.concat([df, prev_dummies], axis=1)
    PREV_COLS = list(prev_dummies.columns)
else:
    PREV_COLS = []
print('Columnas creadas:', PREV_COLS)

Columna previsión: PREVISION
Distribución: {'FONASA_MAI': 1027471, 'FONASA_LIBRE': 18650, 'OTRO': 7202, 'ISAPRE': 5694}
Columnas creadas: ['PREV_FONASA_LIBRE', 'PREV_FONASA_MAI', 'PREV_ISAPRE', 'PREV_OTRO']


### 3.8 ESPECIALIDAD_TOP — top 15 como dummies

In [13]:
esp_col = next((c for c in df.columns if 'ESPECIALIDAD' in c.upper()), None)
print(f'Columna especialidad: {esp_col}')

if esp_col:
    df['_ESP'] = df[esp_col].str.strip().str.upper().fillna('OTRA')
    top15 = df['_ESP'].value_counts().head(15).index.tolist()
    df['ESPECIALIDAD_TOP'] = df['_ESP'].where(df['_ESP'].isin(top15), other='OTRA')
    esp_dummies = pd.get_dummies(df['ESPECIALIDAD_TOP'], prefix='ESP', dtype=int)
    df = pd.concat([df, esp_dummies], axis=1)
    ESP_COLS = list(esp_dummies.columns)
    print(f'Top 15 cubren {df["_ESP"].isin(top15).mean()*100:.1f}% de registros')
else:
    ESP_COLS = []
print('Columnas creadas:', len(ESP_COLS))

Columna especialidad: ESPECIALIDAD_MEDICA
Top 15 cubren 89.7% de registros
Columnas creadas: 16


### 3.9 COD_HOSPITAL — Target Encoding

El hospital es la variable de interés central. Target encoding codifica cada hospital
con su tasa promedio de derivación (suavizada con Laplace smoothing para hospitales
con pocos casos). Esto captura el efecto institucional directo sobre la probabilidad de
derivación, que es precisamente lo que se quiere cuantificar en esta investigación.

In [14]:
hosp_col = next(
    (c for c in df.columns if 'COD_HOSPITAL' in c.upper() or
     ('HOSPITAL' in c.upper() and 'COD' in c.upper())),
    None
)
print(f'Columna hospital: {hosp_col}')

if hosp_col:
    df['_HOSP'] = df[hosp_col].str.strip().fillna('DESCONOCIDO')
    print(f'Hospitales únicos: {df["_HOSP"].nunique()}')

    if TARGET_ENC_AVAILABLE:
        te = TargetEncoder(cols=['_HOSP'], smoothing=10)
        df['HOSP_TARGET_ENC'] = te.fit_transform(df[['_HOSP']], df['TRASLADADO'])['_HOSP']
        HOSP_COL_FINAL = 'HOSP_TARGET_ENC'
    else:
        # Frequency encoding como fallback
        freq_map = df['_HOSP'].value_counts(normalize=True)
        df['HOSP_FREQ_ENC'] = df['_HOSP'].map(freq_map)
        HOSP_COL_FINAL = 'HOSP_FREQ_ENC'

    print(f'Columna hospital codificada: {HOSP_COL_FINAL}')
    print(df[HOSP_COL_FINAL].describe())
else:
    HOSP_COL_FINAL = None

Columna hospital: COD_HOSPITAL
Hospitales únicos: 72
Columna hospital codificada: HOSP_FREQ_ENC
count    1.059017e+06
mean     1.952149e-02
std      9.599095e-03
min      2.925354e-03
25%      1.209612e-02
50%      1.932452e-02
75%      2.470782e-02
max      4.559983e-02
Name: HOSP_FREQ_ENC, dtype: float64


### 3.10 SERVICIO_SALUD — one-hot

In [15]:
ss_col = next((c for c in df.columns if 'SERVICIO_SALUD' in c.upper() or 'SERVICIO' in c.upper()), None)
print(f'Columna servicio salud: {ss_col}')

if ss_col:
    df['_SS'] = df[ss_col].str.strip().str.upper().fillna('DESCONOCIDO')
    print(f'Servicios únicos: {df["_SS"].nunique()}')
    ss_dummies = pd.get_dummies(df['_SS'], prefix='SS', dtype=int)
    df = pd.concat([df, ss_dummies], axis=1)
    SS_COLS = list(ss_dummies.columns)
else:
    SS_COLS = []
print('Columnas creadas:', len(SS_COLS))

Columna servicio salud: SERVICIO_SALUD
Servicios únicos: 30
Columnas creadas: 30


### 3.11 TIPO_PROCEDENCIA — one-hot

In [16]:
proc_col = next((c for c in df.columns if 'PROCEDENCIA' in c.upper()), None)
print(f'Columna tipo procedencia: {proc_col}')

if proc_col:
    df['_PROC'] = df[proc_col].str.strip().str.upper().fillna('DESCONOCIDO')
    print('Valores:', df['_PROC'].value_counts().to_dict())
    proc_dummies = pd.get_dummies(df['_PROC'], prefix='PROC', dtype=int)
    df = pd.concat([df, proc_dummies], axis=1)
    PROC_COLS = list(proc_dummies.columns)
else:
    PROC_COLS = []
print('Columnas creadas:', PROC_COLS)

Columna tipo procedencia: TIPO_PROCEDENCIA
Valores: {'SERVICIO EMERGENCIA (DOMICILIO)': 458569, 'CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)': 368065, 'OTROS HOSPITALES DE LA RED': 74583, 'APS URGENCIA (SAPU, SUR, SUC)': 53321, 'CONSULTA PRIVADA': 23656, 'ESTRATEGIA CRR': 22826, 'APS CONSULTORIO (CESFAM)': 14835, 'OTRAS INSTITUCIONES SALUD (CLÍNICAS PRIVADAS, DE REHABILITAC': 11758, 'OTROS HOSPITALES RED NACIONAL': 11190, 'OTRAS INSTITUCIONES (CÁRCEL, HOGARES DE ANCIANOS, SENAME, EC': 6568, 'CIRUGÍA MAYOR AMBULATORIA (CMA)': 5819, 'CARDIOCIRUGÍA PAGO GRD': 3012, 'HOSPITALIZACIÓN DOMICILIARIA': 2882, 'POSTA RURAL': 1279, 'HOSPITALIZACIÓN DIURNA': 592, 'PLAN DE RESOLUCIÓN LE': 57, 'DESCONOCIDO': 5}
Columnas creadas: ['PROC_APS CONSULTORIO (CESFAM)', 'PROC_APS URGENCIA (SAPU, SUR, SUC)', 'PROC_CARDIOCIRUGÍA PAGO GRD', 'PROC_CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)', 'PROC_CIRUGÍA MAYOR AMBULATORIA (CMA)', 'PROC_CONSULTA PRIVADA', 'PROC_DESCONOCIDO', 'PROC_ESTRAT

### 3.12 TIENE_PROCEDIMIENTO_QUIRURGICO

En CIE-9-MC los procedimientos quirúrgicos corresponden a los capítulos 01–86
(operaciones sobre sistemas orgánicos). Se marca con 1 si al menos uno de los
campos PROCEDIMIENTO1–30 contiene un código numérico en ese rango.

In [17]:
proc_cols = [c for c in [f'PROCEDIMIENTO{i}' for i in range(1, 31)] if c in df.columns]
print(f'Columnas de procedimientos disponibles: {len(proc_cols)}')

def es_quirurgico(val):
    if pd.isna(val):
        return False
    try:
        cod = float(str(val).strip().replace(',', '.').split()[0])
        return 1.0 <= cod <= 86.99
    except (ValueError, IndexError):
        return False

if proc_cols:
    df['TIENE_PROCEDIMIENTO_QUIRURGICO'] = (
        df[proc_cols].apply(lambda row: row.map(es_quirurgico).any(), axis=1)
    ).astype(int)
    pct_quir = df['TIENE_PROCEDIMIENTO_QUIRURGICO'].mean() * 100
    print(f'Pacientes con procedimiento quirúrgico: {pct_quir:.1f}%')
else:
    df['TIENE_PROCEDIMIENTO_QUIRURGICO'] = 0
    print('No se encontraron columnas de procedimientos')

Columnas de procedimientos disponibles: 30
Pacientes con procedimiento quirúrgico: 73.9%


## 4. Análisis Exploratorio (EDA)

Antes de modelar, se analiza el patrón de derivación desde distintas dimensiones institucionales.

### 4.1 Tasa de derivación por hospital (top 20)

In [18]:
if hosp_col:
    tabla_hosp = (
        df.groupby('_HOSP')['TRASLADADO']
        .agg(n_pacientes='count', n_derivados='sum')
        .assign(pct_derivacion=lambda x: 100 * x['n_derivados'] / x['n_pacientes'])
        .sort_values('pct_derivacion', ascending=False)
    )
    top20_hosp = tabla_hosp.head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colores = ['#d62728' if p > tabla_hosp['pct_derivacion'].mean() else '#1f77b4'
               for p in top20_hosp['pct_derivacion']]
    ax.barh(
        top20_hosp.index[::-1],
        top20_hosp['pct_derivacion'].values[::-1],
        color=colores[::-1], edgecolor='none'
    )
    media_global = 100 * df['TRASLADADO'].mean()
    ax.axvline(media_global, color='black', linestyle='--', lw=1.2,
               label=f'Media global ({media_global:.1f}%)')
    ax.set_xlabel('Tasa de derivación (%)')
    ax.set_title('Top 20 hospitales — Tasa de derivación al alta')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'eda_derivacion_por_hospital.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: plots/eda_derivacion_por_hospital.png')
else:
    print('No hay columna COD_HOSPITAL')

Guardado: plots/eda_derivacion_por_hospital.png


### 4.2 Traslados internos según condición de derivación

In [19]:
fig, ax = plt.subplots(figsize=(7, 5))
grupos = [df.loc[df['TRASLADADO'] == 0, 'N_TRASLADOS_INTERNOS'].clip(upper=9),
          df.loc[df['TRASLADADO'] == 1, 'N_TRASLADOS_INTERNOS'].clip(upper=9)]
ax.boxplot(grupos, labels=['No derivado', 'Derivado'],
           patch_artist=True,
           boxprops=dict(facecolor='#aec7e8'),
           medianprops=dict(color='#d62728', linewidth=2))
ax.set_ylabel('N_TRASLADOS_INTERNOS (cap. en 9)')
ax.set_title('Traslados internos según tipo de alta')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_traslados_internos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/eda_traslados_internos.png')

# Estadísticas
for g, lab in [(0, 'No derivado'), (1, 'Derivado')]:
    vals = df.loc[df['TRASLADADO'] == g, 'N_TRASLADOS_INTERNOS']
    print(f'{lab}: media={vals.mean():.2f}, mediana={vals.median():.0f}, max={vals.max()}')

Guardado: plots/eda_traslados_internos.png
No derivado: media=0.22, mediana=0, max=9
Derivado: media=0.40, mediana=0, max=9


### 4.3 Top 10 especialidades con mayor tasa de derivación

In [20]:
if esp_col:
    esp_tasas = (
        df.groupby('_ESP')['TRASLADADO']
        .agg(n='count', derivados='sum')
        .assign(tasa=lambda x: 100 * x['derivados'] / x['n'])
        .query('n >= 100')  # mínimo 100 casos para estabilidad
        .sort_values('tasa', ascending=False)
        .head(10)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(len(esp_tasas)), esp_tasas['tasa'], color='#2ca02c', edgecolor='none')
    ax.set_xticks(range(len(esp_tasas)))
    ax.set_xticklabels(esp_tasas.index, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Tasa de derivación (%)')
    ax.set_title('Top 10 especialidades — mayor tasa de derivación (mín. 100 casos)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'eda_especialidades_derivacion.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: plots/eda_especialidades_derivacion.png')
    print(esp_tasas[['n', 'derivados', 'tasa']].to_string())
else:
    print('No hay columna de especialidad')

Guardado: plots/eda_especialidades_derivacion.png
                                   n  derivados       tasa
_ESP                                                      
MEDICINA DE URGENCIA            4251       1361  32.015996
MEDICINA INTENSIVA ADULTO       9575       2075  21.671018
MEDICINA INTENSIVA PEDIÁTRICA   1760        301  17.102273
MÉDICO GENERAL                  8557       1014  11.849947
CIRUGÍA CARDIOVASCULAR          6154        713  11.585960
CARDIOLOGÍA                    25815       2740  10.613984
NEFROLOGÍA PEDIÁTRICO            135         13   9.629630
CARDIOLOGÍA PEDIÁTRICA           508         48   9.448819
RADIOLOGÍA INTERVENCIONAL        120         11   9.166667
NEUROLOGÍA ADULTO              23417       1900   8.113764


### 4.4 Tasa de derivación por Servicio de Salud

In [21]:
if ss_col:
    ss_tasas = (
        df.groupby('_SS')['TRASLADADO']
        .agg(n='count', derivados='sum')
        .assign(tasa=lambda x: 100 * x['derivados'] / x['n'])
        .sort_values('tasa', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    colores = ['#d62728' if t > ss_tasas['tasa'].mean() else '#1f77b4'
               for t in ss_tasas['tasa']]
    ax.barh(
        ss_tasas.index[::-1],
        ss_tasas['tasa'].values[::-1],
        color=colores[::-1], edgecolor='none'
    )
    ax.axvline(ss_tasas['tasa'].mean(), color='black', linestyle='--', lw=1.2,
               label=f'Media ({ss_tasas["tasa"].mean():.1f}%)')
    ax.set_xlabel('Tasa de derivación (%)')
    ax.set_title('Tasa de derivación por Servicio de Salud')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'eda_derivacion_por_servicio.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: plots/eda_derivacion_por_servicio.png')
    print(ss_tasas[['n', 'derivados', 'tasa']].to_string())
else:
    print('No hay columna SERVICIO_SALUD')

Guardado: plots/eda_derivacion_por_servicio.png
                              n  derivados      tasa
_SS                                                 
ARAUCO                     9468        852  8.998733
VALPARAISO SAN ANTONIO    33928       2679  7.896133
IQUIQUE                   20589       1334  6.479188
METROPOLITANO ORIENTE     39570       2411  6.093000
CHILOÉ                     8832        512  5.797101
ARAUCANÍA SUR             57615       2933  5.090688
METROPOLITANO SUR         58243       2851  4.895009
VIÑA DEL MAR QUILLOTA     50822       2378  4.679076
ÑUBLE                     35213       1610  4.572175
ARAUCANÍA NORTE           20774        900  4.332339
METROPOLITANO SURORIENTE  88541       3675  4.150619
AYSEN                     10184        422  4.143755
LIBERTADOR B. O HIGGINS   44317       1818  4.102263
CONCEPCIÓN                43588       1740  3.991924
VALDIVIA                  23920        923  3.858696
DEL MAULE                 80800       3056  3.78217

### 4.5 Tabla resumen por hospital

In [22]:
if hosp_col:
    tabla_resumen = (
        df.groupby('_HOSP')['TRASLADADO']
        .agg(n_pacientes='count', n_derivados='sum')
        .assign(pct_derivacion=lambda x: (100 * x['n_derivados'] / x['n_pacientes']).round(2))
        .sort_values('pct_derivacion', ascending=False)
        .reset_index()
        .rename(columns={'_HOSP': 'COD_HOSPITAL'})
    )
    print(f'Tabla resumen: {len(tabla_resumen)} hospitales')
    print(tabla_resumen.head(20).to_string(index=False))

Tabla resumen: 72 hospitales
COD_HOSPITAL  n_pacientes  n_derivados  pct_derivacion
      116111         4493          530           11.80
      113180        15417         1729           11.21
      121117         3775          411           10.89
      106100        20663         2041            9.88
      116107         3098          295            9.52
      116110         4644          435            9.37
      128109         6945          645            9.29
      121110         4614          378            8.19
      112101        17183         1350            7.86
      118105         7449          572            7.68
      107101        13926          952            6.84
      111195        10962          711            6.49
      102100        19940         1285            6.44
      119101         3893          244            6.27
      118106         6086          377            6.19
      114101        48291         2883            5.97
      121121         6696          3

## 5. Preparación del dataset final

In [23]:
BASE_FEATURES = [
    'EDAD', 'SEXO_BIN', 'DIAS_HOSPITALIZACION',
    'N_TRASLADOS_INTERNOS', 'N_DIAGNOSTICOS', 'N_PROCEDIMIENTOS',
    'GRD_SEVERIDAD', 'GRD_PESO',
    'TIENE_PROCEDIMIENTO_QUIRURGICO',
]
if HOSP_COL_FINAL:
    BASE_FEATURES.append(HOSP_COL_FINAL)

ALL_FEATURES = BASE_FEATURES + TIPO_ING_COLS + PREV_COLS + ESP_COLS + SS_COLS + PROC_COLS
ALL_FEATURES = [c for c in ALL_FEATURES if c in df.columns]
print(f'Total features: {len(ALL_FEATURES)}')

df_model = df[ALL_FEATURES + ['TRASLADADO']].copy()
df_model = df_model.apply(pd.to_numeric, errors='coerce')

n_antes = len(df_model)
df_model = df_model.dropna()
print(f'Filas eliminadas por NaN: {n_antes - len(df_model):,}')
print(f'Dataset final: {df_model.shape}')
print(f'Tasa de derivación: {df_model["TRASLADADO"].mean()*100:.2f}%')

Total features: 81
Filas eliminadas por NaN: 0
Dataset final: (1059017, 82)
Tasa de derivación: 3.83%


In [24]:
X = df_model[ALL_FEATURES]
y = df_model['TRASLADADO']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape} | derivados: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Test : {X_test.shape}  | derivados: {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

Train: (847213, 81) | derivados: 32,474 (3.83%)
Test : (211804, 81)  | derivados: 8,119 (3.83%)


## 6. Modelado

Se entrenan dos modelos de regresión logística:

- **L2 (Ridge):** regularización cuadrática, mantiene todos los coeficientes pequeños.
  Útil como baseline y para obtener Odds Ratios estables.
- **L1 (Lasso):** regularización absoluta, fuerza coeficientes a exactamente cero.
  Funciona como selección automática de variables.

Ambos usan `class_weight='balanced'` para compensar el desbalance (~3.6% positivos).
Se usa `liblinear` como solver: rápido y estable para datasets grandes con regularización L1/L2.

In [25]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Modelo L2 — baseline
pipe_l2 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        penalty='l2', solver='liblinear',
        class_weight='balanced',
        max_iter=500, random_state=RANDOM_STATE
    ))
])

param_grid_l2 = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}

search_l2 = GridSearchCV(
    pipe_l2, param_grid_l2,
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1
)
print('Entrenando LR L2...')
search_l2.fit(X_train, y_train)
print(f'Mejor C: {search_l2.best_params_["lr__C"]} | ROC-AUC CV: {search_l2.best_score_:.4f}')

Entrenando LR L2...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
# Modelo L1 — selección de variables
pipe_l1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        penalty='l1', solver='liblinear',
        class_weight='balanced',
        max_iter=500, random_state=RANDOM_STATE
    ))
])

param_grid_l1 = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}

search_l1 = GridSearchCV(
    pipe_l1, param_grid_l1,
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1
)
print('Entrenando LR L1 (Lasso)...')
search_l1.fit(X_train, y_train)
print(f'Mejor C: {search_l1.best_params_["lr__C"]} | ROC-AUC CV: {search_l1.best_score_:.4f}')

# Coeficientes en cero (variables eliminadas por Lasso)
coefs_l1 = search_l1.best_estimator_.named_steps['lr'].coef_[0]
n_cero = (coefs_l1 == 0).sum()
print(f'Variables eliminadas por L1: {n_cero}/{len(coefs_l1)} ({100*n_cero/len(coefs_l1):.1f}%)')

## 7. Evaluación

Dado el fuerte desbalance de clases (~3.6% positivos), la curva **Precision-Recall** y
el **Average Precision (AP)** son métricas más informativas que ROC-AUC, ya que esta última
puede ser engañosamente alta en datasets desbalanceados.

In [ ]:
def evaluar_modelo(modelo, X_te, y_te, nombre, umbral=0.5):
    y_prob = modelo.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= umbral).astype(int)
    auc = roc_auc_score(y_te, y_prob)
    ap  = average_precision_score(y_te, y_prob)
    f1  = f1_score(y_te, y_pred)
    print(f'\n{"="*55}')
    print(f'Modelo: {nombre}')
    print(f'ROC-AUC: {auc:.4f}  |  Avg Precision (PR-AUC): {ap:.4f}  |  F1 (positivo): {f1:.4f}')
    print(classification_report(y_te, y_pred, target_names=['NO DERIVADO', 'DERIVADO'], digits=4))
    return y_prob, auc, ap, f1

prob_l2, auc_l2, ap_l2, f1_l2 = evaluar_modelo(search_l2, X_test, y_test, 'LR L2 (Ridge)')
prob_l1, auc_l1, ap_l1, f1_l1 = evaluar_modelo(search_l1, X_test, y_test, 'LR L1 (Lasso)')

In [ ]:
# Curvas ROC y Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

modelos_eval = [
    ('LR L2 (Ridge)', prob_l2, auc_l2, ap_l2),
    ('LR L1 (Lasso)', prob_l1, auc_l1, ap_l1),
]

for nombre, prob, auc, ap in modelos_eval:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, lw=2, label=f'{nombre} (AUC={auc:.3f})')
    prec, rec, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(rec, prec, lw=2, label=f'{nombre} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1], 'k--', lw=0.8)
axes[0].set_xlabel('Tasa de Falsos Positivos'); axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color='k', linestyle='--', lw=0.8, label=f'Baseline ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_pr_traslado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/roc_pr_traslado.png')

In [ ]:
# Matriz de confusión — LR L2
y_pred_l2 = (prob_l2 >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_l2)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NO DERIVADO', 'DERIVADO']).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Matriz de Confusión — LR L2 (umbral=0.5)')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_traslado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/confusion_traslado.png')

In [ ]:
# Tabla comparativa de coeficientes L2 vs L1
lr_l2 = search_l2.best_estimator_.named_steps['lr']
lr_l1 = search_l1.best_estimator_.named_steps['lr']

df_coefs = pd.DataFrame({
    'Feature'    : ALL_FEATURES,
    'Coef_L2'    : lr_l2.coef_[0],
    'Coef_L1'    : lr_l1.coef_[0],
})
df_coefs['Abs_L2'] = df_coefs['Coef_L2'].abs()
df_coefs = df_coefs.sort_values('Abs_L2', ascending=False).reset_index(drop=True)

print('Top 20 features por magnitud de coeficiente L2:')
print(df_coefs.head(20)[['Feature','Coef_L2','Coef_L1']].to_string(index=False))

# Variables no nulas en L1
vars_seleccionadas = df_coefs[df_coefs['Coef_L1'] != 0]['Feature'].tolist()
print(f'\nVariables seleccionadas por L1: {len(vars_seleccionadas)}/{len(ALL_FEATURES)}')

In [ ]:
# Gráfico comparativo coeficientes L2 (top 20)
top_n = 20
df_plot = df_coefs.head(top_n).sort_values('Coef_L2')
colors  = ['#d62728' if c > 0 else '#1f77b4' for c in df_plot['Coef_L2']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(df_plot['Feature'], df_plot['Coef_L2'], color=colors, edgecolor='none', height=0.7)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coeficiente estandarizado (log-odds)')
ax.set_title(f'Top {top_n} features — LR L2 (coeficientes estandarizados)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'coeficientes_traslado_l2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/coeficientes_traslado_l2.png')

## 8. Odds Ratios con Intervalos de Confianza al 95%

Los **Odds Ratios (OR)** son la exponencial de los coeficientes logísticos. Un OR > 1 indica
que la variable aumenta las probabilidades de derivación; OR < 1 las reduce.

Los IC 95% se calculan mediante **bootstrap paramétrico** sobre una muestra del conjunto
de entrenamiento (n=5.000, 150 iteraciones). Esto es una aproximación; para inferencia
estadística formal se recomienda usar modelos sin regularización (e.g., `statsmodels`).

> **Nota sobre regularización y OR:** la penalización L2 contrae los coeficientes hacia 0,
> lo que sesga los OR hacia 1.0. Los OR deben interpretarse como indicadores de dirección
> y magnitud relativa, no como estimaciones puntuales no sesgadas.

In [ ]:
def bootstrap_or_ci(
    pipeline, X_tr, y_tr, feature_names,
    n_bootstrap=150, sample_size=5000, ci=0.95
):
    """Calcula OR con IC bootstrap sobre una muestra."""
    scaler = pipeline.named_steps['scaler']
    lr     = pipeline.named_steps['lr']
    C_best = lr.C

    X_scaled = scaler.transform(X_tr)
    n = min(sample_size, len(X_scaled))
    idx_all = np.arange(len(X_scaled))
    y_arr = y_tr.values if hasattr(y_tr, 'values') else np.array(y_tr)

    boot_coefs = []
    rng = np.random.default_rng(RANDOM_STATE)

    for _ in range(n_bootstrap):
        idx = rng.choice(idx_all, size=n, replace=True)
        lr_b = LogisticRegression(
            penalty=lr.penalty, C=C_best, solver=lr.solver,
            class_weight='balanced', max_iter=300, random_state=RANDOM_STATE
        )
        try:
            lr_b.fit(X_scaled[idx], y_arr[idx])
            boot_coefs.append(lr_b.coef_[0])
        except Exception:
            pass

    if not boot_coefs:
        return None

    boot_arr = np.array(boot_coefs)
    alpha    = (1 - ci) / 2
    ci_lower = np.percentile(boot_arr, 100 * alpha, axis=0)
    ci_upper = np.percentile(boot_arr, 100 * (1 - alpha), axis=0)
    coefs    = lr.coef_[0]

    return pd.DataFrame({
        'Feature'       : feature_names,
        'Coeficiente'   : coefs,
        'OR'            : np.exp(coefs),
        'OR_IC95_lower' : np.exp(ci_lower),
        'OR_IC95_upper' : np.exp(ci_upper),
    }).sort_values('OR', ascending=False).reset_index(drop=True)

print('Calculando Odds Ratios con bootstrap (puede tardar ~1 min)...')
df_or = bootstrap_or_ci(search_l2.best_estimator_, X_train, y_train, ALL_FEATURES)
print('Listo.')
print(df_or.head(20).to_string(index=False))

In [ ]:
# Gráfico Forest plot de Odds Ratios (top 25 por distancia de OR respecto a 1)
if df_or is not None:
    df_or_plot = df_or.copy()
    df_or_plot['dist_de_1'] = (df_or_plot['OR'] - 1).abs()
    df_or_plot = df_or_plot.sort_values('dist_de_1', ascending=False).head(25).sort_values('OR')

    fig, ax = plt.subplots(figsize=(8, 9))
    y_pos = range(len(df_or_plot))
    colors_or = ['#d62728' if o > 1 else '#1f77b4' for o in df_or_plot['OR']]

    ax.barh(y_pos, df_or_plot['OR'] - 1, left=1, color=colors_or, alpha=0.7, height=0.6)
    ax.errorbar(
        df_or_plot['OR'], y_pos,
        xerr=[
            df_or_plot['OR'] - df_or_plot['OR_IC95_lower'],
            df_or_plot['OR_IC95_upper'] - df_or_plot['OR']
        ],
        fmt='none', color='black', capsize=3, lw=1.2
    )
    ax.axvline(1, color='black', linestyle='--', lw=1.0)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_or_plot['Feature'], fontsize=8)
    ax.set_xlabel('Odds Ratio (IC 95% bootstrap)')
    ax.set_title('Forest plot — OR para derivación al alta (LR L2)')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'odds_ratios_traslado.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: plots/odds_ratios_traslado.png')

### Interpretación clínica de los Odds Ratios principales

**Variables que aumentan la probabilidad de derivación (OR > 1):**
- **N_TRASLADOS_INTERNOS:** Fuerte predictor — un paciente ya trasladado internamente
  múltiples veces es muy probable que requiera derivación al alta. Puede reflejar
  complejidad clínica no manejable en el establecimiento.
- **GRD_SEVERIDAD alta:** Pacientes más graves en un hospital de baja complejidad
  tienen más chances de ser derivados a un centro con mayor capacidad resolutiva.
- **HOSP_TARGET_ENC:** Cuando este coeficiente es alto, captura el efecto institucional:
  hospitales con historial de alta tasa de derivación tienden a derivar más.

**Variables que reducen la probabilidad de derivación (OR < 1):**
- **TIPO_INGRESO_PROGRAMADA:** Ingresos electivos suelen corresponder a procedimientos
  planificados que se completan en el mismo establecimiento.
- **GRD_PESO alto:** Paradójicamente, casos muy complejos en hospitales de alta
  complejidad son menos derivados porque el propio establecimiento los resuelve.
- **TIENE_PROCEDIMIENTO_QUIRURGICO:** Los pacientes operados suelen egresar al domicilio
  o al alta voluntaria, no a derivación.

> **Advertencia de causalidad:** Los OR no implican causalidad.  
> La interpretación es correlacional y debe contextualizarse con el funcionamiento real de la red asistencial.

## 9. Análisis por hospital: tasa real vs. probabilidad predicha

In [ ]:
if hosp_col:
    # Predicciones en todo el dataset (train + test)
    prob_all = search_l2.best_estimator_.predict_proba(X)[:, 1]
    df_analisis = df_model.copy()
    df_analisis['PROB_PREDICHA'] = prob_all

    # Recuperar COD_HOSPITAL original para el análisis
    df_analisis['_HOSP'] = df.loc[df_model.index, '_HOSP'].values

    hosp_comp = (
        df_analisis.groupby('_HOSP')
        .agg(
            n_pacientes=('TRASLADADO', 'count'),
            n_derivados=('TRASLADADO', 'sum'),
            tasa_real=('TRASLADADO', 'mean'),
            prob_predicha_media=('PROB_PREDICHA', 'mean')
        )
        .assign(
            tasa_real_pct=lambda x: (x['tasa_real'] * 100).round(2),
            prob_predicha_pct=lambda x: (x['prob_predicha_media'] * 100).round(2),
            diferencia=lambda x: (x['prob_predicha_pct'] - x['tasa_real_pct']).round(2)
        )
        .sort_values('tasa_real_pct', ascending=False)
        .reset_index()
        .rename(columns={'_HOSP': 'COD_HOSPITAL'})
    )

    print('Tasa real vs probabilidad predicha por hospital (top 20):')
    print(hosp_comp[['COD_HOSPITAL','n_pacientes','n_derivados',
                      'tasa_real_pct','prob_predicha_pct','diferencia']].head(20).to_string(index=False))

In [ ]:
if hosp_col:
    # Scatter tasa real vs predicha
    hosp_plot = hosp_comp[hosp_comp['n_pacientes'] >= 200]

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        hosp_plot['tasa_real_pct'],
        hosp_plot['prob_predicha_pct'],
        c=np.log1p(hosp_plot['n_pacientes']),
        cmap='viridis', alpha=0.75, s=60, edgecolors='none'
    )
    lim_max = max(hosp_plot[['tasa_real_pct','prob_predicha_pct']].max()) * 1.1
    ax.plot([0, lim_max], [0, lim_max], 'k--', lw=1.0, label='Línea perfecta')
    plt.colorbar(sc, ax=ax, label='log(n pacientes)')
    ax.set_xlabel('Tasa real de derivación (%)')
    ax.set_ylabel('Prob. predicha media (%)')
    ax.set_title('Real vs Predicho por hospital (mín. 200 pacientes)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'hospitales_real_vs_predicho.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: plots/hospitales_real_vs_predicho.png')

    # Hospitales con mayor diferencia (subestimados / sobreestimados)
    print('\nHospitales más subestimados por el modelo (real >> predicho):')
    print(hosp_comp.nsmallest(5, 'diferencia')[['COD_HOSPITAL','tasa_real_pct','prob_predicha_pct','diferencia']].to_string(index=False))
    print('\nHospitales más sobreestimados por el modelo (predicho >> real):')
    print(hosp_comp.nlargest(5, 'diferencia')[['COD_HOSPITAL','tasa_real_pct','prob_predicha_pct','diferencia']].to_string(index=False))

## 10. Exportación de resultados

In [ ]:
# Modelo final
model_path = MODELS_DIR / 'modelo_traslado.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({
        'model'           : search_l2.best_estimator_,
        'features'        : ALL_FEATURES,
        'best_params'     : search_l2.best_params_,
        'roc_auc_test'    : auc_l2,
        'avg_prec_test'   : ap_l2,
        'f1_test'         : f1_l2,
    }, f)
print(f'Modelo guardado en {model_path}')

# Odds Ratios
if df_or is not None:
    or_path = DATA_DIR / 'odds_ratios_traslado.csv'
    df_or.to_csv(or_path, index=False)
    print(f'OR guardados en {or_path}')

# Análisis por hospital
if hosp_col:
    hosp_path = DATA_DIR / 'hospitales_derivacion.csv'
    hosp_comp.to_csv(hosp_path, index=False)
    print(f'Tabla hospitales guardada en {hosp_path}')

# Coeficientes
coef_path = DATA_DIR / 'coeficientes_traslado.csv'
df_coefs.drop(columns='Abs_L2').to_csv(coef_path, index=False)
print(f'Coeficientes guardados en {coef_path}')

## 11. Resumen metodológico y limitaciones

### Resumen

| Aspecto | Decisión |
|---|---|
| Variable objetivo | Binaria: TIPOALTA contiene 'DERIVACI' |
| Exclusiones | Fallecidos (mecanismo de egreso diferente) + SEXO DESCONOCIDO |
| Split | 80/20 estratificado |
| Balanceo | class_weight='balanced' en ambos modelos |
| Validación cruzada | StratifiedKFold 5-fold |
| Métricas principales | ROC-AUC + Average Precision (PR-AUC) |
| Modelo recomendado | LR L2 (coeficientes más estables para OR) |
| Encoding hospital | Target Encoding (smoothing=10) o Frequency Encoding |
| IC para OR | Bootstrap paramétrico (n=150, muestra=5.000 — aproximación) |

### Limitaciones

1. **GRD asignado al alta:** Los campos de severidad y peso GRD no están disponibles
   al momento del ingreso, lo que limita el uso prospectivo del modelo.

2. **Datos de corte transversal:** El dataset corresponde a un año calendario.
   No se capturan tendencias temporales ni cambios en la red asistencial.

3. **Target encoding con posible leakage:** Al calcular el target encoding de COD_HOSPITAL
   sobre el dataset completo, hay una contaminación leve del test set. En producción,
   este encoding debe calcularse estrictamente sobre el train fold en cada split de CV.

4. **Sesgo de selección institucional:** Hospitales de mayor complejidad resolutiva
   derivan menos, pero también atienden casos más graves. El modelo puede confundir
   efecto institucional con efecto de casemix. Para separar ambos efectos se requieren
   técnicas de ajuste por severidad (ej. estandarización indirecta).

5. **IC de OR aproximados:** Los intervalos de confianza calculados via bootstrap
   con regularización L2 están sesgados hacia 1.0. Para publicación formal, usar
   `statsmodels.api.Logit` sin regularización sobre variables preseleccionadas.

### Extensiones sugeridas
- Modelos multinomial para distinguir tipos de derivación (dentro del servicio vs. red nacional)
- Análisis de heterogeneidad de efectos por Servicio de Salud (interacciones)
- Modelo de doble robustez (AIPW) para estimar el efecto causal del nivel de complejidad del hospital
- Validación temporal: entrenar en 2022–2023, evaluar en 2024